# 04 Fine-tuning
This notebook performs few-shot fine-tuning (N={10, 50, 100}) for all three conditions (GlocalIB, GlocalIB beta=0, MLM) and saves the results.

In [ ]:
import sys
import os
import json
import torch
import numpy as np
from torch.optim import AdamW
from sklearn.metrics import f1_score
from transformers import RobertaTokenizer, RobertaForMaskedLM

sys.path.append("..")
from src.data import load_ecthr, sample_few_shot
from src.model import GlocalIBModel, DocumentClassifier

# --- CONFIGURATION ---
N_LIST        = [10, 50, 100]
SEEDS         = [0, 1, 2, 3, 4]
EPOCHS        = 10
LR            = 2e-5
RESULTS_DIR   = "../results"
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Checkpoint Paths (Verify these exist after pre-training!)
CONDITIONS = {
    "glocal_ib":    ("../checkpoints/glocal_ib_epoch5.pt",    "glocal"),
    "glocal_beta0": ("../checkpoints/glocal_beta0_epoch5.pt", "glocal"),
    "mlm":          ("../checkpoints/mlm/mlm_final",          "mlm"),
}

In [ ]:
def load_encoder(path, ckpt_type, device):
    if ckpt_type == "glocal":
        model = GlocalIBModel(device=device)
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        return model.encoder, model.tokenizer
    else:
        mlm = RobertaForMaskedLM.from_pretrained(path)
        tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
        return mlm.roberta.to(device), tokenizer

def train_one_few_shot(encoder, tokenizer, train_split, test_split, n, seed, device):
    few_shot = sample_few_shot(train_split, n, seed)
    clf = DocumentClassifier(encoder, tokenizer, device=device)
    optimizer = AdamW(clf.parameters(), lr=LR)
    criterion = torch.nn.BCELoss()

    clf.train()
    for epoch in range(EPOCHS):
        for ex in few_shot:
            optimizer.zero_grad()
            probs = clf([ex["text"]])
            
            labels = torch.zeros(1, 10).to(device)
            for l in ex["labels"]:
                if l < 10: labels[0][l] = 1.0
                
            loss = criterion(probs, labels)
            loss.backward()
            optimizer.step()
            
    # Eval
    clf.eval()
    preds, targets = [], []
    with torch.no_grad():
        for ex in test_split:
            p = clf([ex["text"]]).cpu().numpy()[0]
            preds.append((p >= 0.5).astype(int))
            
            t = np.zeros(10, dtype=int)
            for l in ex["labels"]:
                if l < 10: t[l] = 1
            targets.append(t)
            
    return f1_score(np.array(targets), np.array(preds), average="macro")

In [ ]:
print("Loading dataset...")
dataset = load_ecthr()
all_results = {}

for cond, (path, c_type) in CONDITIONS.items():
    if not os.path.exists(path):
        print(f"Skipping {cond}, checkpoint not found at {path}")
        continue
        
    print(f"\nEvaluating Condition: {cond}")
    encoder, tokenizer = load_encoder(path, c_type, DEVICE)
    all_results[cond] = {}
    
    for n in N_LIST:
        scores = []
        for seed in SEEDS:
            f1 = train_one_few_shot(encoder, tokenizer, dataset["train"], dataset["test"], n, seed, DEVICE)
            scores.append(f1)
            print(f"  N={n} | Seed={seed} | Macro-F1: {f1:.4f}")
        all_results[cond][str(n)] = scores

with open(f"{RESULTS_DIR}/finetuning_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\nFine-tuning Complete. Results saved.")